In [6]:
import torch
import torch.nn as nn

In [7]:
#Mô phỏng mạng Feed Forward và các "đường tắt" (Residual Connection) để tín hiệu không bị hao hụt
#Định nghĩa mạng Feed Forward (FFN)
#Gồm 2 phép biến đổi tuyến tính và ReLU ở giữa
class FeedForward(nn.Module):
    def __init__(self, d_model=512, d_ff=2048):
        super().__init__()
        # Mở rộng từ 512 lên 2048 chiều
        self.linear1 = nn.Linear(d_model, d_ff)
        self.relu = nn.ReLU()
        # Ánh xạ ngược về lại 512 chiều
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(self.relu(self.linear1(x)))

#Khối Encoder hoàn chỉnh (Encoder Block)
class EncoderBlock(nn.Module):
    def __init__(self, d_model=512):
        super().__init__()
        #Dùng MultiHeadAttention, giả lập bằng Linear 
        self.self_attention = nn.Linear(d_model, d_model) 
        self.norm1 = nn.LayerNorm(d_model) # Chuẩn hóa lớp 1
        
        self.ffn = FeedForward(d_model)
        self.norm2 = nn.LayerNorm(d_model) # Chuẩn hóa lớp 2

    def forward(self, x):
        #Self-Attention + Residual Connection (Thang máy)
        # x + Sublayer(x)
        attention_res = self.norm1(x + self.self_attention(x))
        
        #Feed Forward + Residual Connection
        #Giúp tinh chỉnh lại biểu diễn của từng token
        final_out = self.norm2(attention_res + self.ffn(attention_res))
        
        return final_out

In [8]:
# Giả đinh Input là 1 câu gồm 10 tokens, mỗi token 512 chiều (từ 01.Embedding)
input_data = torch.randn(1, 10, 512)
print(f"INPUT , Kích thước: {input_data.shape} (1 câu, 10 từ, 512 chiều)")

# Khởi tạo khối Encoder
encoder_block = EncoderBlock(d_model=512)

# Thực thi toàn bộ khối
output_data = encoder_block(input_data)

print(f"OUTPUT , Kích thước: {output_data.shape}")
print(f"Giá trị vector (5 phần tử đầu của token thứ 1):\n{output_data[0, 0, :5].detach().numpy()}")

INPUT , Kích thước: torch.Size([1, 10, 512]) (1 câu, 10 từ, 512 chiều)
OUTPUT , Kích thước: torch.Size([1, 10, 512])
Giá trị vector (5 phần tử đầu của token thứ 1):
[ 1.0927345  -1.447391    0.70648754  0.07690179  1.0472571 ]
